In [ ]:
from libsvmdata import fetch_libsvm
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import os

from trainer import train

In [ ]:
X, y = fetch_libsvm("abalone_scale")
X.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y.reshape(-1, 1), test_size=0.2, random_state=42)

scaler = StandardScaler()
y_train_scaled = scaler.fit_transform(y_train)
y_test_scaled = scaler.transform(y_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).unsqueeze(-1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32).unsqueeze(-1)

BATCH_SIZE = X_train_tensor.shape[0]

train_data_real = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
test_data_real = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)
train_data_complex = DataLoader(TensorDataset(X_train_tensor.type(torch.complex64), y_train_tensor.type(torch.complex64)), batch_size=BATCH_SIZE, shuffle=True)
test_data_complex = DataLoader(TensorDataset(X_test_tensor.type(torch.complex64), y_test_tensor.type(torch.complex64)), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def make_complex64(layers):
    for layer in layers:
        layer.weight.data = layer.weight.data.type(torch.complex64)
        layer.bias.data = layer.bias.data.type(torch.complex64)

def normal_init_weights(layers):
    mean = 0
    std = 1e-5
    for layer in layers:
        nn.init.normal_(layer.weight.data, mean, std)
        if layer.bias is not None:
            nn.init.normal_(layer.bias.data, mean, std)

In [ ]:
class RegressionNN(nn.Module):
    def __init__(self, input_size, hidden_size, complex_weights=False):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 1)
        self.activation = nn.Tanh()
        
        if complex_weights:
            make_complex64([self.fc1, self.fc2])
        normal_init_weights([self.fc1, self.fc2])
        

    def forward(self, x):
        x_in = x.squeeze(-1)
        h = self.activation(self.fc1(x_in))
        out = self.fc2(h)
        return out.unsqueeze(-1)

In [ ]:
alpha = 0.0

def batch_to_tensors(batch):
    return batch[0], batch[1], torch.tensor([])

def complex_mse_loss(d, y, model):
    return (d - y).abs().square().sum() + alpha * sum(torch.norm(p)**2 for p in model.parameters())

def loss(model, signal_batch):
    x, d, _ = batch_to_tensors(signal_batch)
    return complex_mse_loss(d, model(x), model)

def r2_score_complex(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mean_true = np.mean(y_true)
    total_sum_squares = np.sum(abs(y_true - mean_true) ** 2)
    residual_sum_squares = np.sum(abs(y_true - y_pred) ** 2)
    r2 = 1 - (residual_sum_squares / total_sum_squares)

    return r2

def criterion(model, dataset):
    ds = []
    ys = []
    for batch in dataset:
        x, d, _ = batch_to_tensors(batch)
        ds.append(d.squeeze(0, 1).detach())
        ys.append(model(x).squeeze(0, 1).detach())
    d = torch.cat(ds, dim=0)
    y = torch.cat(ys, dim=0)
    return r2_score_complex(d, y)

Модель с комплексными параметрами:

In [ ]:
hidden_size = 10
complex_weights = True
train_type = 'mnm_lev_marq'
random_seed = 1

In [ ]:
learning_curve, best_criterion = train(model_complex, train_data_complex, loss, criterion, {}, batch_to_tensors=batch_to_tensors, validate_dataset=train_data_complex, test_dataset=val_data_complex, train_type='mnm_lev_marq', save_path='results/', exp_name='_first', chunk_num=1, save_every=1)